In [117]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sktechno710/collabspace-events/collabspace_events.csv
/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/aggregate_churn.csv


In [118]:
df = pd.read_csv('/kaggle/input/datasets/sktechno710/collabspace-events/collabspace_events.csv')
churn_df = pd.read_csv('/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/aggregate_churn.csv')

In [119]:
df

,user_id,event_date,event_type
0,1000,2025-01-06,login
1,1000,2025-01-09,send_message
2,1000,2025-01-21,login
3,1000,2025-01-24,upload_file
4,1000,2025-01-25,send_message
...,...,...,...
13781,1499,2025-06-11,send_message
13782,1499,2025-06-15,send_message
13783,1499,2025-06-15,login
13784,1499,2025-06-25,upload_file


In [120]:
churn_df

,user_id,days_since_last_event,last_event_date,IsChurned
0,1000,6 days,2025-04-11,1
1,1001,5 days,2025-06-24,0
2,1002,8 days,2025-04-10,1
3,1003,6 days,2025-06-13,0
4,1004,5 days,2025-03-24,1
...,...,...,...,...
495,1495,15 days,2025-06-29,0
496,1496,3 days,2025-06-29,0
497,1497,1 days,2025-04-06,1
498,1498,1 days,2025-06-27,0


In [121]:
df['event_date']=pd.to_datetime(df['event_date'])
churn_df['last_event_date']=pd.to_datetime(churn_df['last_event_date'])

In [122]:
current_date = df['event_date'].max()

In [123]:
# calculating new feature values for user_id 1000
df[df['user_id']==1000]['user_id'].count()

np.int64(21)

In [124]:
def total_events_cal(row):
    return df[df['user_id']==row['user_id']]['user_id'].count()

def account_age_days_cal(row):
    return current_date - df[df['user_id']==row['user_id']]['event_date'].min()

In [125]:
churn_df['total_events'] = churn_df.apply(total_events_cal, axis=1)
churn_df['account_age_day'] = churn_df.apply(account_age_days_cal,axis=1)
churn_df

,user_id,days_since_last_event,last_event_date,IsChurned,total_events,account_age_day
0,1000,6 days,2025-04-11,1,21,174 days
1,1001,5 days,2025-06-24,0,29,179 days
2,1002,8 days,2025-04-10,1,11,164 days
3,1003,6 days,2025-06-13,0,27,178 days
4,1004,5 days,2025-03-24,1,13,179 days
...,...,...,...,...,...,...
495,1495,15 days,2025-06-29,0,31,177 days
496,1496,3 days,2025-06-29,0,31,168 days
497,1497,1 days,2025-04-06,1,21,175 days
498,1498,1 days,2025-06-27,0,26,174 days


In [126]:
def event_freq_cal(row):
    return row['total_events']/row['account_age_day'].days
churn_df['event_frequency'] = churn_df.apply(event_freq_cal, axis=1)
churn_df

,user_id,days_since_last_event,last_event_date,IsChurned,total_events,account_age_day,event_frequency
0,1000,6 days,2025-04-11,1,21,174 days,0.120690
1,1001,5 days,2025-06-24,0,29,179 days,0.162011
2,1002,8 days,2025-04-10,1,11,164 days,0.067073
3,1003,6 days,2025-06-13,0,27,178 days,0.151685
4,1004,5 days,2025-03-24,1,13,179 days,0.072626
...,...,...,...,...,...,...,...
495,1495,15 days,2025-06-29,0,31,177 days,0.175141
496,1496,3 days,2025-06-29,0,31,168 days,0.184524
497,1497,1 days,2025-04-06,1,21,175 days,0.120000
498,1498,1 days,2025-06-27,0,26,174 days,0.149425


In [127]:
# days_since_last_event is actually time between latest two events of a user id so renaming it 
churn_df.rename(columns = {'days_since_last_event':'time_between_last_two_event'},inplace=True)
churn_df

,user_id,time_between_last_two_event,last_event_date,IsChurned,total_events,account_age_day,event_frequency
0,1000,6 days,2025-04-11,1,21,174 days,0.120690
1,1001,5 days,2025-06-24,0,29,179 days,0.162011
2,1002,8 days,2025-04-10,1,11,164 days,0.067073
3,1003,6 days,2025-06-13,0,27,178 days,0.151685
4,1004,5 days,2025-03-24,1,13,179 days,0.072626
...,...,...,...,...,...,...,...
495,1495,15 days,2025-06-29,0,31,177 days,0.175141
496,1496,3 days,2025-06-29,0,31,168 days,0.184524
497,1497,1 days,2025-04-06,1,21,175 days,0.120000
498,1498,1 days,2025-06-27,0,26,174 days,0.149425


In [128]:
# calculating core_actions_ratio
df[df['user_id']==1001][df[df['user_id']==1001]['event_type']=='send_message']['user_id'].count()

np.int64(11)

In [129]:
def core_actions_ratio_cal(row):
    id = row['user_id']
    # count for create project event for an id
    a = df[df['user_id']==id][df[df['user_id']==id]['event_type']=='create_project']['user_id'].count()
    # count for upload file event for an id
    b = df[df['user_id']==id][df[df['user_id']==id]['event_type']=='upload_file']['user_id'].count()
    return a+b
churn_df['core_actions_ratio_cal']=churn_df.apply(core_actions_ratio_cal, axis=1)
churn_df

,user_id,time_between_last_two_event,last_event_date,IsChurned,total_events,account_age_day,event_frequency,core_actions_ratio_cal
0,1000,6 days,2025-04-11,1,21,174 days,0.120690,3
1,1001,5 days,2025-06-24,0,29,179 days,0.162011,5
2,1002,8 days,2025-04-10,1,11,164 days,0.067073,2
3,1003,6 days,2025-06-13,0,27,178 days,0.151685,7
4,1004,5 days,2025-03-24,1,13,179 days,0.072626,4
...,...,...,...,...,...,...,...,...
495,1495,15 days,2025-06-29,0,31,177 days,0.175141,11
496,1496,3 days,2025-06-29,0,31,168 days,0.184524,10
497,1497,1 days,2025-04-06,1,21,175 days,0.120000,7
498,1498,1 days,2025-06-27,0,26,174 days,0.149425,6


In [145]:
churn_df[churn_df['user_id']==1000]['account_age_day'].dt.days

0    174
Name: account_age_day, dtype: int64

In [140]:
int(churn_df[churn_df['user_id']==1000]['time_between_last_two_event'].str.split()[0][0])

6

In [149]:
# Converting account_age_day to numerical value
def convert_to_num(row):
    return row['account_age_day'].days
    
churn_df['account_age_day'] = churn_df.apply(convert_to_num, axis=1)
churn_df

,user_id,time_between_last_two_event,last_event_date,IsChurned,total_events,account_age_day,event_frequency,core_actions_ratio_cal
0,1000,6,2025-04-11,1,21,174,0.120690,3
1,1001,5,2025-06-24,0,29,179,0.162011,5
2,1002,8,2025-04-10,1,11,164,0.067073,2
3,1003,6,2025-06-13,0,27,178,0.151685,7
4,1004,5,2025-03-24,1,13,179,0.072626,4
...,...,...,...,...,...,...,...,...
495,1495,1,2025-06-29,0,31,177,0.175141,11
496,1496,3,2025-06-29,0,31,168,0.184524,10
497,1497,1,2025-04-06,1,21,175,0.120000,7
498,1498,1,2025-06-27,0,26,174,0.149425,6


In [150]:
def convert_to_num(row):
    return row['time_between_last_two_event'].split()[0][0]
    
churn_df['time_between_last_two_event'] = churn_df.apply(convert_to_num, axis=1)
churn_df

,user_id,time_between_last_two_event,last_event_date,IsChurned,total_events,account_age_day,event_frequency,core_actions_ratio_cal
0,1000,6,2025-04-11,1,21,174,0.120690,3
1,1001,5,2025-06-24,0,29,179,0.162011,5
2,1002,8,2025-04-10,1,11,164,0.067073,2
3,1003,6,2025-06-13,0,27,178,0.151685,7
4,1004,5,2025-03-24,1,13,179,0.072626,4
...,...,...,...,...,...,...,...,...
495,1495,1,2025-06-29,0,31,177,0.175141,11
496,1496,3,2025-06-29,0,31,168,0.184524,10
497,1497,1,2025-04-06,1,21,175,0.120000,7
498,1498,1,2025-06-27,0,26,174,0.149425,6


In [151]:
y = churn_df['IsChurned']
x = churn_df[['time_between_last_two_event','total_events','account_age_day','event_frequency','core_actions_ratio_cal']]

In [152]:
# Train test Split, conserving the class imbalance in both set
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [153]:
x_train

,time_between_last_two_event,total_events,account_age_day,event_frequency,core_actions_ratio_cal
306,7,28,178,0.157303,11
402,1,34,179,0.189944,11
49,9,15,178,0.084270,3
198,5,25,162,0.154321,8
433,3,33,175,0.188571,7
...,...,...,...,...,...
394,6,24,178,0.134831,6
50,2,17,179,0.094972,3
400,1,39,178,0.219101,15
10,6,29,178,0.162921,7


In [154]:
y_train

306    0
402    0
49     1
198    0
433    0
      ..
394    0
50     1
400    0
10     0
153    0
Name: IsChurned, Length: 400, dtype: int64

### Baseline Model Training - Random Forest without handling class imbalance

In [155]:
# Training RF without class imbalance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train,y_train)

RandomForestClassifier(random_state=42)

In [158]:
y_pred = model.predict(x_test)

In [160]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8900

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93        77
           1       0.77      0.74      0.76        23

    accuracy                           0.89       100
   macro avg       0.85      0.84      0.84       100
weighted avg       0.89      0.89      0.89       100



In [175]:
from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC AUC: {auc_score}")

ROC AUC: 0.8370976849237719


In [167]:
model.predict_proba(x_test)

array([[1.  , 0.  ],
       [0.99, 0.01],
       [0.98, 0.02],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [0.92, 0.08],
       [0.86, 0.14],
       [1.  , 0.  ],
       [0.99, 0.01],
       [0.09, 0.91],
       [0.72, 0.28],
       [0.89, 0.11],
       [0.24, 0.76],
       [1.  , 0.  ],
       [0.02, 0.98],
       [0.01, 0.99],
       [1.  , 0.  ],
       [0.71, 0.29],
       [0.32, 0.68],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [0.96, 0.04],
       [0.01, 0.99],
       [0.81, 0.19],
       [1.  , 0.  ],
       [0.92, 0.08],
       [0.9 , 0.1 ],
       [0.06, 0.94],
       [0.3 , 0.7 ],
       [0.01, 0.99],
       [0.21, 0.79],
       [0.97, 0.03],
       [0.  , 1.  ],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [0.67, 0.33],
       [0.91, 0.09],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [1.  , 0.  ],
       [0.1 , 0.9 ],
       [1.  , 0.  ],
       [0.98, 0.02],
       [0.03, 0.97],
       [0.96,

### Logistic Regression Model without handling imbalance

In [176]:
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression()
model2.fit(x_train,y_train)
y_pred2 = model2.predict(x_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred2):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred2))
auc_score = roc_auc_score(y_test, y_pred2)
print(f"ROC AUC: {auc_score}")

Accuracy: 0.9000

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.94        77
           1       0.81      0.74      0.77        23

    accuracy                           0.90       100
   macro avg       0.87      0.84      0.85       100
weighted avg       0.90      0.90      0.90       100

ROC AUC: 0.8435911914172783


### SMOTE for Class Balance

In [177]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [178]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Inspect original class distribution
print(f"Original training distribution: {Counter(y_train)}")

# Initialize SMOTE
smote = SMOTE(sampling_strategy='auto', random_state=42)

# Resample only the training data
X_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)

# Inspect balanced class distribution
print(f"Balanced training distribution: {Counter(y_train_balanced)}")

Original training distribution: Counter({0: 308, 1: 92})
Balanced training distribution: Counter({0: 308, 1: 308})


In [179]:
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression()
model2.fit(x_train,y_train)
y_pred2 = model2.predict(x_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred2):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred2))
auc_score = roc_auc_score(y_test, y_pred2)
print(f"ROC AUC: {auc_score}")

Accuracy: 0.9000

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.94        77
           1       0.81      0.74      0.77        23

    accuracy                           0.90       100
   macro avg       0.87      0.84      0.85       100
weighted avg       0.90      0.90      0.90       100

ROC AUC: 0.8435911914172783


In [180]:
# Training RF without class imbalance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train,y_train)

y_pred = model.predict(x_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC AUC: {auc_score}")

Accuracy: 0.8900

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93        77
           1       0.77      0.74      0.76        23

    accuracy                           0.89       100
   macro avg       0.85      0.84      0.84       100
weighted avg       0.89      0.89      0.89       100

ROC AUC: 0.8370976849237719
